# Weight Decomposition

Decomposes weight matrices using SVD and related methods: identifying shared
substructure, weight clustering, spectral analysis, and norm distributions.

**References:**
- Sharma et al. (2023) "The Truth is in There: Improving Reasoning with Layer-Selective Rank Reduction"
- Hsu et al. (2022) "Language Model Decomposition"

## Why This Matters

Composition weight decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_decomposition import (
    weight_svd_decomposition,
    weight_shared_substructure,
    weight_clustering,
    spectral_weight_analysis,
    weight_norm_distribution,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")

In [ ]:
# 1. SVD decomposition of weight matrices
for l in range(cfg.n_layers):
    for comp in ['attn', 'mlp']:
        svd = weight_svd_decomposition(model, layer=l, component=comp)
        ranks = ', '.join(f"{k}={v:.0f}" for k, v in svd['effective_ranks'].items())
        print(f"L{l} {comp}: compression={svd['compression_ratio']:.2%}, ranks=[{ranks}]")

In [ ]:
# 2. Shared substructure across layers
shared = weight_shared_substructure(model)
print(f"Most similar layers: {shared['most_similar_layers']}")
print(f"Shared subspace dim: {shared['shared_subspace_dim']}")
print(f"\nCross-layer similarity:")
for i in range(cfg.n_layers):
    row = ' '.join(f"{shared['cross_layer_similarity'][i,j]:+.3f}" for j in range(cfg.n_layers))
    print(f"  L{i}: {row}")
print(f"\nWeight norms: {shared['layer_weight_norms'].tolist()}")

In [ ]:
# 3. Weight clustering
clust = weight_clustering(model, n_clusters=3)
print(f"Cluster sizes: {clust['cluster_sizes']}")
for (layer, comp), label in sorted(clust['labels'].items()):
    print(f"  L{layer} {comp}: cluster {label}")
print(f"Within-cluster variance: {clust['within_cluster_variance'].tolist()}")

In [ ]:
# 4. Spectral analysis
spec = spectral_weight_analysis(model)
print(f"Worst conditioned: L{spec['worst_conditioned'][0]} {spec['worst_conditioned'][1]}")
for (l, name), cn in sorted(spec['condition_numbers'].items()):
    sn = spec['spectral_norms'][(l, name)]
    rd = spec['rank_deficiency'][(l, name)]
    print(f"  L{l} {name:>8}: cond={cn:.1f}, spectral_norm={sn:.4f}, rank_def={rd:.2%}")

In [ ]:
# 5. Weight norm distribution
norms = weight_norm_distribution(model)
print(f"Total model norm: {norms['total_norm']:.4f}")
for l in range(cfg.n_layers):
    ratio = norms['norm_ratio_attn_mlp'][l]
    print(f"  L{l}: attn={norms['attn_norms_by_layer'][l]:.4f}, "
          f"mlp={norms['mlp_norms_by_layer'][l]:.4f}, ratio={ratio:.2f}")